# XGBOOST

El modelo debe predecir a los aprobados ya que con esta información, la universidad diseñará los programas de becas por el curso. El Recall es importante porque te ayudará a evitar que el modelo se pierda estudiantes que merecían ser considerados para las becas, lo cual afectaría el diseño del programa.(evitar falsos negativos).

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, roc_auc_score

from xgboost import XGBClassifier

data = pd.read_csv('dataset_tdah_completo (1).csv', sep=';')

data.head()

,id,edad,genero,grado,condicion_social,edah_h1,edah_h2,edah_h3,edah_h4,edah_h5,...,conners_inatencion_tscore,conners_oposicion_tscore,conners_adhd_index_tscore,promedio_notas,prom_atencion,prom_hiperactividad,tdah_presente,tdah_probabilidad,nivel_riesgo_academico,split
0,1,12,M,7,0,2,2,3,2,3,...,65.7,54.7,73.7,10.2,2.81,1.78,1,0.760,Medio,train
1,2,14,F,9,1,1,1,1,0,0,...,41.4,35.8,30.0,18.8,0.52,0.00,0,0.313,Bajo,train
2,3,13,M,8,0,0,0,1,1,1,...,48.8,37.5,51.1,18.3,0.93,0.35,0,0.087,Bajo,train
3,4,12,F,7,0,1,1,1,0,1,...,33.5,46.0,44.1,12.7,0.77,0.49,0,0.044,Bajo,train
4,5,14,M,9,3,1,1,1,1,1,...,46.5,54.2,52.2,10.0,0.84,0.01,0,0.178,Medio,train


In [2]:
data.shape

(500, 40)

In [3]:
data['tdah_presente'].value_counts()

tdah_presente
0    343
1    157
Name: count, dtype: int64

In [4]:
columnas_eliminar = [
    'id',
    'tdah_presente',
    'tdah_probabilidad',
    'split',

    'edah_h1','edah_h2','edah_h3','edah_h4','edah_h5',
    'edah_da1','edah_da2','edah_da3','edah_da4','edah_da5',

    'edah_tc1','edah_tc2','edah_tc3','edah_tc4','edah_tc5',
    'edah_tc6','edah_tc7','edah_tc8','edah_tc9','edah_tc10',

    'edah_h_total',
    'edah_da_total',
    'edah_tdah_total',
    'edah_tc_total',

    'conners_hiperact_tscore',
    'conners_inatencion_tscore',
    'conners_oposicion_tscore',
    'conners_adhd_index_tscore',

    'prom_atencion',
    'prom_hiperactividad'
]

X = data.drop(columns=columnas_eliminar)

y = data['tdah_presente']

X = pd.get_dummies(X, drop_first=True)

In [5]:
X = data.drop(columns=columnas_eliminar)

y = data['tdah_presente']

In [6]:
X = pd.get_dummies(
    X,
    columns=['genero', 'nivel_riesgo_academico'],
    drop_first=True
)

In [7]:
print(X.shape)
X.head()

(500, 7)


,edad,grado,condicion_social,promedio_notas,genero_M,nivel_riesgo_academico_Bajo,nivel_riesgo_academico_Medio
0,12,7,0,10.2,True,False,True
1,14,9,1,18.8,False,True,False
2,13,8,0,18.3,True,True,False
3,12,7,0,12.7,False,True,False
4,14,9,3,10.0,True,False,True


In [8]:
from sklearn.model_selection import train_test_split

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.25,
    random_state=42,
    stratify=y_trainval
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(300, 7)
(100, 7)
(100, 7)


In [9]:
from xgboost import XGBClassifier

xgb_0 = XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    eval_metric='logloss',
    max_depth=4,
    n_estimators=100,
    learning_rate=0.05
)

xgb_0.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [10]:
y_pred_train = xgb_0.predict(X_train)
y_pred_val = xgb_0.predict(X_val)
y_pred_test = xgb_0.predict(X_test)

In [11]:
from sklearn.metrics import confusion_matrix

print("TRAIN")
print(confusion_matrix(y_train, y_pred_train))

print("\nVALIDATION")
print(confusion_matrix(y_val, y_pred_val))

print("\nTEST")
print(confusion_matrix(y_test, y_pred_test))

TRAIN
[[205   1]
 [  2  92]]

VALIDATION
[[64  4]
 [ 6 26]]

TEST
[[63  6]
 [ 1 30]]


In [12]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.98      0.91      0.95        69
           1       0.83      0.97      0.90        31

    accuracy                           0.93       100
   macro avg       0.91      0.94      0.92       100
weighted avg       0.94      0.93      0.93       100



In [13]:
from sklearn.metrics import confusion_matrix

print("TRAIN")
print(confusion_matrix(y_train, y_pred_train))

print("VALIDATION")
print(confusion_matrix(y_val, y_pred_val))

print("TEST")
print(confusion_matrix(y_test, y_pred_test))

TRAIN
[[205   1]
 [  2  92]]
VALIDATION
[[64  4]
 [ 6 26]]
TEST
[[63  6]
 [ 1 30]]


In [14]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

           0       0.98      0.91      0.95        69
           1       0.83      0.97      0.90        31

    accuracy                           0.93       100
   macro avg       0.91      0.94      0.92       100
weighted avg       0.94      0.93      0.93       100



In [15]:
# Separar características y target
X = data.drop('tdah_presente', axis=1)
y = data['tdah_presente']

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score

In [17]:
# Dividir el dataset
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval)

In [18]:
X_train.shape, X_val.shape, X_test.shape , X_trainval.shape

((300, 39), (100, 39), (100, 39), (400, 39))

In [19]:
from xgboost import XGBClassifier

## Modelo 0 - XGboost valores por defecto

### Modelar

In [20]:
#Entrenamiento TRAIN
xgb_0 = XGBClassifier(
    random_state=42,
    objective= 'binary:logistic',
    enable_categorical=True # Habilitar el manejo de características categóricas (experimental)
    )

# Identificar las columnas de tipo 'object' en X_train y convertirlas a 'category'
categorical_cols = X_train.select_dtypes(include='object').columns
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')

#objective='multi:softmax',   # Para clasificación múltiple
#num_class=3,
xgb_0.fit(X_train, y_train)

C:\Users\Kenneth\AppData\Local\Temp\ipykernel_3136\986003040.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include='object').columns


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [21]:
#Ver estabilidad CV
from sklearn.model_selection import cross_val_score

scores = cross_val_score(xgb_0, X_train, y_train, cv=5, scoring='recall')  # o 'f1', 'roc_auc', etc.

print("CV recall scores:", scores)
print("Mean CV recall:", scores.mean())
print("Std CV recall (estabilidad):", scores.std())

CV recall scores: [1.         0.94736842 1.         1.         1.        ]
Mean CV recall: 0.9894736842105264
Std CV recall (estabilidad): 0.02105263157894739


In [22]:
# Generar las predicciones con TRAIN
y_pred_train = xgb_0.predict(X_train)

In [23]:
# Generar las predicciones con VALID
# Identificar las columnas de tipo 'object' en X_val y convertirlas a 'category'
categorical_cols_val = X_val.select_dtypes(include='object').columns
for col in categorical_cols_val:
    X_val[col] = X_val[col].astype('category')

y_pred_val = xgb_0.predict(X_val)

C:\Users\Kenneth\AppData\Local\Temp\ipykernel_3136\806480350.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_val = X_val.select_dtypes(include='object').columns


### Métricas y visualizacion

In [24]:
# Métricas
from sklearn.metrics import classification_report, confusion_matrix, recall_score

In [25]:
print("Matrix confusion de TRAIN:")  ##OVERFITI
print(confusion_matrix(y_train, y_pred_train))

print("Matrix confusion de VAL:")
print(confusion_matrix(y_val, y_pred_val))

Matrix confusion de TRAIN:
[[206   0]
 [  0  94]]
Matrix confusion de VAL:
[[68  0]
 [ 0 32]]


In [26]:
print("Métricas en TRAIN:")
print(classification_report(y_train, y_pred_train))

# Predicción en VALID
print("Métricas en VALID:")
print(classification_report(y_val, y_pred_val))

Métricas en TRAIN:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       206
           1       1.00      1.00      1.00        94

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300

Métricas en VALID:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        68
           1       1.00      1.00      1.00        32

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [27]:
#Hiperparámetros por defecto
default_params = xgb_0.get_params()
print(default_params)

{'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': None, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': None, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': None, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': None, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': None, 'tree_method': None, 'validate_parameters': None, 'verbosity': None}


## Modelo 1

#### Modelar

In [28]:
xgb01 = XGBClassifier(
    objective      ='binary:logistic',   # Para clasificación binaria
    eval_metric    = ['logloss','auc'], # Métrica de monitoreo (cuando detenerse)## LOGLOOSS MIDE EL ERROR ###AUC----QUE TAN CERCA ESTÁ DE LA REALIDAD, MIENTRAS MAS CERCA A 1, ##LE HACE CASO AL AUC, SI SE INVIERTE HACE CASO  AL PRIMERO, USUALMENTE AL SEGVUNDO
    random_state=42,
    enable_categorical=True # Add this line to enable categorical feature handling
    #scale_pos_weight=2.7
    )

In [29]:
# Definir la cuadrícula de hiperparámetros
# param_grid = {
#     'n_estimators': np.arange(50, 300, 50),
#     'max_depth': np.arange(3, 10),
#     'learning_rate': [0.01, 0.1, 0.2], #()np.linspace(0.01, 0.1, 0.2, 0.3, 10),
#     'subsample': np.linspace(0.6, 1.0, 5),
#     'colsample_bytree': np.linspace(0.6, 1.0, 5)
# }


# param_grid = {
#     'n_estimators': np.arange(100, 500, 50),  # Rango más amplio y flexible
#     'max_depth': [3, 4, 5, 6, 7, 8],          # No muy grandes para evitar overfitting
#     'learning_rate': [0.01, 0.05, 0.1, 0.2],  # Más granularidad para aprendizaje
#     'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],   # Evita overfitting si es < 1
#     'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],  # Igual que subsample
#     'gamma': [0, 0.1, 0.3, 0.5, 1],           # Regularización para poda de ramas débiles
#     'reg_alpha': [0, 0.1, 0.5, 1],            # L1 regularization (sparse)
#     'reg_lambda': [0.5, 1, 1.5, 2]            # L2 regularization
# }


param_grid = {
    'n_estimators': np.arange(50, 300, 50),   ###CANTIDAD DE ARBOLES
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1],  ##TAMAÑO DEL PASO,
    'subsample': np.linspace(0.6, 1.0, 4), ##FRACCION DE MUESTRA DE REGISTROS
    'colsample_bytree': np.linspace(0.6, 1.0, 4), ##FEACCIONES DE MUESRA  POR COLUMNA
    'reg_alpha': [0, 0.1, 1, 5],         # L1 regularización ###PENALIZACION DE LOS  ERRORES
    'reg_lambda': [1, 5, 10],            # L2 regularización ###PENALIZACIOND DE LOS ERRORES
    'min_child_weight': [1, 3, 5, 10],   # Controla el sobreajuste ###
}

In [30]:
#Entrenar el modelo con los hiperparámrtros y los datos de TRAIN
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


random_search = RandomizedSearchCV(
        estimator  = xgb01,
        param_distributions = param_grid ,
        cv = skf,
        n_iter = 50, ##RAMDONIZD VA TOMAR SOLO 50
        n_jobs = -1,
        verbose = 8,
        scoring = 'recall',
        random_state = 42
      )


random_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': array([0.6 ..., 1. ]), 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 4, ...], 'min_child_weight': [1, 3, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'recall'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can 

In [31]:
#Ver la estabilidad del modelo (promedio y desviacion) y los mejores hiperparámetros
import pandas as pd
cv_results = pd.DataFrame(random_search.cv_results_)
cv_results.head(4)
#cv_results[['mean_test_score', 'std_test_score', 'params']]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_subsample,param_reg_lambda,param_reg_alpha,param_n_estimators,param_min_child_weight,param_max_depth,...,param_colsample_bytree,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.056936,0.004544,0.012528,0.006149,1.000000,1,0.0,250,3,3,...,0.733333,"{'subsample': 1.0, 'reg_lambda': 1, 'reg_alpha...",0.944444,1.0,1.000000,1.0,1.0,0.988889,0.022222,1
1,0.048886,0.009072,0.008015,0.001824,0.600000,10,5.0,150,10,3,...,0.600000,"{'subsample': 0.6, 'reg_lambda': 10, 'reg_alph...",0.944444,1.0,0.947368,1.0,1.0,0.978363,0.026516,47
2,0.064865,0.009508,0.012050,0.006992,0.866667,10,5.0,250,5,6,...,1.000000,"{'subsample': 0.8666666666666667, 'reg_lambda'...",0.944444,1.0,1.000000,1.0,1.0,0.988889,0.022222,1
3,0.033228,0.008019,0.006606,0.001093,0.600000,10,5.0,100,5,5,...,1.000000,"{'subsample': 0.6, 'reg_lambda': 10, 'reg_alph...",0.944444,1.0,1.000000,1.0,1.0,0.988889,0.022222,1


In [32]:
cv_results.sort_values(by='rank_test_score', ascending=True).head(4)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_subsample,param_reg_lambda,param_reg_alpha,param_n_estimators,param_min_child_weight,param_max_depth,...,param_colsample_bytree,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.056936,0.004544,0.012528,0.006149,1.000000,1,0.0,250,3,3,...,0.733333,"{'subsample': 1.0, 'reg_lambda': 1, 'reg_alpha...",0.944444,1.0,1.0,1.0,1.0,0.988889,0.022222,1
2,0.064865,0.009508,0.012050,0.006992,0.866667,10,5.0,250,5,6,...,1.000000,"{'subsample': 0.8666666666666667, 'reg_lambda'...",0.944444,1.0,1.0,1.0,1.0,0.988889,0.022222,1
3,0.033228,0.008019,0.006606,0.001093,0.600000,10,5.0,100,5,5,...,1.000000,"{'subsample': 0.6, 'reg_lambda': 10, 'reg_alph...",0.944444,1.0,1.0,1.0,1.0,0.988889,0.022222,1
4,0.022461,0.008007,0.009618,0.005931,0.600000,5,0.0,50,10,6,...,0.600000,"{'subsample': 0.6, 'reg_lambda': 5, 'reg_alpha...",0.944444,1.0,1.0,1.0,1.0,0.988889,0.022222,1


In [33]:
print("CV recall scores:", scores)  ##NP.INT64---250 ARBOLES CON EL MEJOR RESULTADO
print("Mejor recall promedio (CV):", random_search.best_score_)
print("Mejores hiperparámetros:", random_search.best_params_)
#xgb01_random_search =  random_search.best_estimator_
best_params = random_search.best_params_

CV recall scores: [1.         0.94736842 1.         1.         1.        ]
Mejor recall promedio (CV): 0.9888888888888889
Mejores hiperparámetros: {'subsample': np.float64(1.0), 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': np.int64(250), 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': np.float64(0.7333333333333333)}


In [34]:



# Entrenamos con early stopping usando VALIDACIÓN EXTERNA (val) ##el auc busca el mas alto

xgb01 = XGBClassifier(
    **best_params,
    eval_metric           = ['auc','logloss'], # Métrica de monitoreo (cuando detenerse) USA LA SEGUNDA DE LA LISTA
    objective             = 'binary:logistic',##LOGLOSS OBTIENE DEL VAL Y EL AUC DEL TRAIN
    random_state          = 42,
    early_stopping_rounds = 10,
    enable_categorical=True # Explicitly add this parameter
    )

xgb01.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=True
)

#Guardar la mejor iteración
best_iter = xgb01.best_iteration
print("Mejor iteración:", best_iter)

[0]	validation_0-auc:1.00000	validation_0-logloss:0.57487	validation_1-auc:1.00000	validation_1-logloss:0.57957
[1]	validation_0-auc:1.00000	validation_0-logloss:0.53472	validation_1-auc:1.00000	validation_1-logloss:0.53810
[2]	validation_0-auc:1.00000	validation_0-logloss:0.49886	validation_1-auc:1.00000	validation_1-logloss:0.50109
[3]	validation_0-auc:1.00000	validation_0-logloss:0.46537	validation_1-auc:1.00000	validation_1-logloss:0.46733
[4]	validation_0-auc:1.00000	validation_0-logloss:0.43502	validation_1-auc:1.00000	validation_1-logloss:0.43676
[5]	validation_0-auc:1.00000	validation_0-logloss:0.40737	validation_1-auc:1.00000	validation_1-logloss:0.40892
[6]	validation_0-auc:1.00000	validation_0-logloss:0.38206	validation_1-auc:1.00000	validation_1-logloss:0.38345
[7]	validation_0-auc:1.00000	validation_0-logloss:0.35922	validation_1-auc:1.00000	validation_1-logloss:0.36021
[8]	validation_0-auc:1.00000	validation_0-logloss:0.33776	validation_1-auc:1.00000	validation_1-logloss:

[20]	validation_0-auc:1.00000	validation_0-logloss:0.17284	validation_1-auc:1.00000	validation_1-logloss:0.17216
[21]	validation_0-auc:1.00000	validation_0-logloss:0.16392	validation_1-auc:1.00000	validation_1-logloss:0.16327
[22]	validation_0-auc:1.00000	validation_0-logloss:0.15552	validation_1-auc:1.00000	validation_1-logloss:0.15489
[23]	validation_0-auc:1.00000	validation_0-logloss:0.14762	validation_1-auc:1.00000	validation_1-logloss:0.14700
[24]	validation_0-auc:1.00000	validation_0-logloss:0.14017	validation_1-auc:1.00000	validation_1-logloss:0.13957
[25]	validation_0-auc:1.00000	validation_0-logloss:0.13314	validation_1-auc:1.00000	validation_1-logloss:0.13257
[26]	validation_0-auc:1.00000	validation_0-logloss:0.12651	validation_1-auc:1.00000	validation_1-logloss:0.12596
[27]	validation_0-auc:1.00000	validation_0-logloss:0.12025	validation_1-auc:1.00000	validation_1-logloss:0.11972
[28]	validation_0-auc:1.00000	validation_0-logloss:0.11434	validation_1-auc:1.00000	validation_1

In [35]:
#Mejor score
print("Mejor score:", xgb01.best_score) # siempre la ultima metrica definida
print(min(xgb01.evals_result()['validation_1']['logloss'])) #si es el logloss se busca el menor

Mejor score: 0.02066722692921758
0.02066722692921758


In [36]:
#Accesar a los valores
xgb01.evals_result() #['validation_1']['logloss']

{'validation_0': OrderedDict([('auc',
               [1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1.0,
                1

In [37]:
# Generar las predicciones con TRAIN
y_pred_train = xgb01.predict(X_train)
# Generar las predicciones con VALID
y_pred_val = xgb01.predict(X_val)

#### Métricas y visualizacion

In [38]:
# Métricas
from sklearn.metrics import classification_report, confusion_matrix, recall_score

In [39]:
print("Matrix confusion de TRAIN:")
print(confusion_matrix(y_train, y_pred_train))

print("Matrix confusion de VAL:")
print(confusion_matrix(y_val, y_pred_val))

Matrix confusion de TRAIN:
[[206   0]
 [  0  94]]
Matrix confusion de VAL:
[[68  0]
 [ 0 32]]


In [40]:
print("Métricas en TRAIN:")
print(classification_report(y_train, y_pred_train))


print("Métricas en VALID:")
print(classification_report(y_val, y_pred_val))

Métricas en TRAIN:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       206
           1       1.00      1.00      1.00        94

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300

Métricas en VALID:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        68
           1       1.00      1.00      1.00        32

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



## Evaluar el modelo final

In [41]:
params_sin_estimators = best_params.copy()
params_sin_estimators.pop('n_estimators', None)

final_model = XGBClassifier(
    **params_sin_estimators,
    n_estimators=best_iter + 1,  # # Para evitar overfitting
    random_state=42,
    enable_categorical=True  # Add this parameter to handle categorical features
)

# Convert 'object' columns in X_trainval to 'category' dtype
categorical_cols_trainval = X_trainval.select_dtypes(include='object').columns
for col in categorical_cols_trainval:
    X_trainval[col] = X_trainval[col].astype('category')

final_model.fit(X_trainval, y_trainval)

C:\Users\Kenneth\AppData\Local\Temp\ipykernel_3136\1434841921.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_trainval = X_trainval.select_dtypes(include='object').columns


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,np.float64(0.7333333333333333)
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datas

#### Méricas con TEST

In [42]:
# Convert 'object' columns in X_test to 'category' dtype
categorical_cols_test = X_test.select_dtypes(include='object').columns
for col in categorical_cols_test:
    X_test[col] = X_test[col].astype('category')

# Generar las predicciones con TEST
y_pred_test = final_model.predict(X_test)

C:\Users\Kenneth\AppData\Local\Temp\ipykernel_3136\3766158234.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_test = X_test.select_dtypes(include='object').columns


In [43]:
# Métricas
from sklearn.metrics import classification_report, confusion_matrix, recall_score

In [44]:
print("Matrix confusion de TEST:")
print(confusion_matrix(y_test, y_pred_test))

Matrix confusion de TEST:
[[69  0]
 [ 0 31]]


In [45]:
print("Métricas en TEST:")
print(classification_report(y_test, y_pred_test))

Métricas en TEST:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        69
           1       1.00      1.00      1.00        31

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [46]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test, y_pred_test)

1.0

## Ejemplo de Predicción

In [47]:
# Ejemplo de una persona a identificar
# The example_person DataFrame must have the same features as X_trainval
# Let's create a hypothetical person's data based on the features the model expects.
# Using X_trainval.columns to get the expected feature names.

# Create a dictionary for a hypothetical person with representative values
# We'll use the column names from X_trainval and fill with sample data.
# For simplicity, we can take a row from the original 'data' or X_trainval as a template.

# Get the feature names from X_trainval
expected_features = X_trainval.columns.tolist()

# Create a dictionary with sample data for a hypothetical person
# For numeric columns, provide sensible numbers.
# For categorical columns ('genero', 'nivel_riesgo_academico', 'split'), use string values that exist in your dataset.

# Example values (you might want to adjust these to represent a specific case)
# Let's take the mean for numerical features and mode for categorical features from the training data
example_data_dict = {}
for col in expected_features:
    # Check if the column is categorical (either 'object' or 'category' dtype)
    if X_trainval[col].dtype == 'object' or isinstance(X_trainval[col].dtype, pd.CategoricalDtype):
        example_data_dict[col] = [X_trainval[col].mode()[0]] # Use mode for categorical
    else:
        # Otherwise, it's numerical, use mean
        example_data_dict[col] = [X_trainval[col].mean()] # Use mean for numerical

# Manually adjust a few values to represent a person we want to predict for
# For instance, if 'edad' and 'promedio_notas' are key features
# Let's assume a 13-year-old male with average notes and academic risk.
example_data_dict['id'] = [999] # Placeholder ID
example_data_dict['edad'] = [13]
example_data_dict['genero'] = ['M']
example_data_dict['nivel_riesgo_academico'] = ['Bajo']
example_data_dict['promedio_notas'] = [15.0]
example_data_dict['split'] = ['train'] # This column was part of X and often used to split data, but for a single prediction, it might not be relevant to the model itself. However, it needs to be present.


example_person = pd.DataFrame(example_data_dict)

# Convert 'object' columns in example_person to 'category' dtype, similar to training data
categorical_cols_example = example_person.select_dtypes(include='object').columns
for col in categorical_cols_example:
    example_person[col] = example_person[col].astype('category')

# Realizar la predicción
prediction = final_model.predict(example_person)

# Mostrar la predicción
result = "Aprobado" if prediction[0] == 1 else "No Aprobado"
print(f"La persona con las características dadas será: {result}")

La persona con las características dadas será: No Aprobado


C:\Users\Kenneth\AppData\Local\Temp\ipykernel_3136\305487746.py:42: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_example = example_person.select_dtypes(include='object').columns


In [48]:
# # Definir la cuadrícula de hiperparámetros
# param_grid = {
#     'n_estimators': [20, 30, 50],                   # Número de árboles
#     'learning_rate': [0.001, 0.01, 0.1, 0.2],                 # Tasa de aprendizaje
#     'max_depth': [3, 4, 5, 6, 7],                      # Profundidad máxima de un árbol
#     #'min_child_weight': [1, 2, 3, 4],                  # Mínimo peso de los hijos
#     'colsample_bytree': [0.6, 0.8, 1.0],               # Proporción de columnas utilizadas por árbol
#     'gamma': [0, 0.1, 0.2, 0.3],                        # Reducción de la función de pérdida
#     #'scale_pos_weight': [1, 2, 3],                     # Peso para clases desbalanceadas
#     'objective': ['binary:logistic'],                   # Función de objetivo
#     'eval_metric': ['logloss','auc', 'error'],                   # Métrica de evaluación
#     #'booster': ['gbtree', 'gblinear']                   # Tipo de booster
#     'reg_alpha': [0.01, 0.1, 1],  # L1 regularization
#     'reg_lambda': [0.01, 0.1, 1],  # L2 regularization
#     'subsample': [0.7, 0.8, 0.9]  # Para evitar que el modelo aprenda demasiado de un subconjunto
# }


# param_grid = {
#     'n_estimators': [50, 100, 200],               # Menos árboles
#     'learning_rate': [0.001,0.01, 0.05, 0.1],           # Tasas de aprendizaje más bajas
#     'max_depth': np.arange(5, 30, 2),                    # Profundidad de árbol más baja
#     'min_child_weight': [1, 2],             # Ponderación de hijos mínima
#     'colsample_bytree': [0.6, 0.8],         # Menos columnas por árbol
#     'gamma': [0, 0.1],                      # Control de pérdida
#     'reg_alpha': [0.1, 0.5, 1],                  # Aumento de regularización L1
#     'reg_lambda': [0.1, 0.5, 1],                 # Aumento de regularización L2
#     'subsample': [0.7, 0.8, 0.9],                # Fracción de datos por árbol
#     'objective': ['binary:logistic'],       # Función de objetivo
#     'eval_metric': ['logloss', 'auc'],      # Métricas de evaluación,
#     'scale_pos_weight' : [2]
# }

# # xgboost_model = xgb.XGBClassifier(
# #     n_estimators=100,               # Número de árboles
# #     max_depth=5,                    # Profundidad máxima del árbol
# #     learning_rate=0.1,              # Tasa de aprendizaje
# #     subsample=0.8,                  # Proporción de muestras a usar para cada árbol
# #     colsample_bytree=0.8,           # Proporción de características a usar para cada árbol
# #     gamma=0,                        # Complejidad de la penalización
# #     reg_alpha=0,                    # Regularización L1
# #     reg_lambda=1,                   # Regularización L2
# #     use_label_encoder=False,        # Usar encoder de etiquetas
# #     eval_metric='logloss'           # Métrica de evaluación
# # )